# BigQuery Comic Processor

This notebook automates the extraction of comic metadata from images stored in Google Cloud Storage and populates a BigQuery table with the details.

### Requirements:
- A Google Cloud Project
- GCS bucket with comic images (.jpg)
- Vertex AI and BigQuery APIs enabled


In [ ]:
# @title Configuration
PROJECT_ID = 'ndy-814-20251212174826' # @param {type:"string"}
SOURCE_BUCKET = 'gs://sample_comics' # @param {type:"string"}
LOCATION = 'us-central1' # @param {type:"string"}
DATASET_ID = 'comic_processing' # @param {type:"string"}
TABLE_ID = 'comic_details' # @param {type:"string"}
CONNECTION_ID = 'vertex-ai-connection-comics' # @param {type:"string"}


In [ ]:
# @title Install Libraries
!pip install --upgrade google-cloud-aiplatform google-cloud-bigquery google-cloud-storage google-cloud-bigquery-connection


In [ ]:
# @title Enable APIs
!gcloud services enable aiplatform.googleapis.com \
                        bigquery.googleapis.com \
                        bigqueryconnection.googleapis.com \
                        storage.googleapis.com \
                        --project {PROJECT_ID}


In [ ]:
# @title Create BigQuery Connection and Set IAM
import json

# Create the BigQuery connection
!bq mk --connection --location={LOCATION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE {CONNECTION_ID}

# Get the service account for the connection using JSON format for robust parsing
output = !bq --format=json show --connection --project_id={PROJECT_ID} --location={LOCATION} {CONNECTION_ID}
sa_email = None

try:
    connection_info = json.loads("".join(output))
    sa_email = connection_info.get("cloudResource", {}).get("serviceAccountId")
except Exception as e:
    print(f"Error parsing connection info: {e}")
    # Fallback to manual parsing if JSON fails
    for line in output:
        if "serviceAccountId" in line:
            # Clean up potential quotes, braces, and whitespace
            sa_email = line.split(": ")[1].strip().strip('"').strip('}').strip(',')
            break

if sa_email:
    print(f"Connection Service Account: {sa_email}")
    # Assign IAM roles to the service account
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
        --member=serviceAccount:{sa_email} \
        --role=roles/aiplatform.user --quiet
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
        --member=serviceAccount:{sa_email} \
        --role=roles/storage.objectViewer --quiet
else:
    print("Error: Could not retrieve connection service account.")


In [ ]:
# @title Initialize BigQuery Table
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

# Create Dataset
dataset_ref = client.dataset(DATASET_ID)
try:
    client.get_dataset(dataset_ref)
    print(f"Dataset {DATASET_ID} already exists.")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = LOCATION
    client.create_dataset(dataset)
    print(f"Created dataset {DATASET_ID}.")

# Create Table
schema = [
    bigquery.SchemaField("comic_file_name", "STRING"),
    bigquery.SchemaField("comic_length", "INTEGER"),
    bigquery.SchemaField("publisher", "STRING"),
    bigquery.SchemaField("title", "STRING"),
    bigquery.SchemaField("issue_number", "INTEGER"),
    bigquery.SchemaField("date_published", "DATE"),
    bigquery.SchemaField("summary", "STRING"),
]

table_ref = dataset_ref.table(TABLE_ID)
try:
    client.get_table(table_ref)
    print(f"Table {TABLE_ID} already exists.")
except Exception:
    table = bigquery.Table(table_ref, schema=schema)
    client.create_table(table)
    print(f"Created table {TABLE_ID}.")


In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel, Part, Tool
import json

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Model for all tasks using Gemini 2.5 Flash
# Removed search_tool definition as GoogleSearchRetrieval is not available.
model = GenerativeModel("gemini-2.5-flash")

def get_comic_details(file_uri):
    # Analyze the file and search for metadata using a single model call
    image_part = Part.from_uri(file_uri, mime_type="image/jpeg")

    prompt = f"""
    Analyze this comic page and search for information about the comic shown in {file_uri}.
    Identify the:
    - Approximated number of pages in the full comic
    - Publisher
    - Title
    - Issue Number
    - Date Published (YYYY-MM-DD format)
    - Summary (less than 30 words)

    Return as JSON:
    {{
        "comic_length": int,
        "publisher": "string",
        "title": "string",
        "issue_number": int,
        "date_published": "YYYY-MM-DD",
        "summary": "string"
    }}
    """

    try:
        response = model.generate_content([image_part, prompt])
        json_text = response.text.strip('`').replace('json', '').strip()
        data = json.loads(json_text)
    except Exception as e:
        print(f"Error in Gemini 2.5 Flash processing: {e}")
        data = {
            "comic_length": 1,
            "publisher": "Unknown",
            "title": "Unknown",
            "issue_number": 0,
            "date_published": "2000-01-01",
            "summary": "Could not retrieve details."
        }

    return data

In [ ]:
# @title Process Comics and Insert into BigQuery
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)
bucket_name = SOURCE_BUCKET.replace("gs://", "")
blobs = storage_client.list_blobs(bucket_name)

rows_to_insert = []

for blob in blobs:
    if blob.name.lower().endswith(('.jpg', '.jpeg', '.png')):
        file_uri = f"gs://{bucket_name}/{blob.name}"
        print(f"Processing {file_uri}...")

        details = get_comic_details(file_uri)

        row = {
            "comic_file_name": blob.name,
            "comic_length": details.get("comic_length"),
            "publisher": details.get("publisher"),
            "title": details.get("title"),
            "issue_number": details.get("issue_number"),
            "date_published": details.get("date_published"),
            "summary": details.get("summary"),
        }
        rows_to_insert.append(row)

if rows_to_insert:
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
    errors = client.insert_rows_json(table_id, rows_to_insert)
    if not errors:
        print(f"Successfully inserted {len(rows_to_insert)} rows.")
    else:
        print(f"Errors occurred: {errors}")
else:
    print("No comics found to process.")
